# Study 1 — 01c: Validation Reliability

Expert validation κ between auto-coded and manually coded traces. Confusion matrix analysis and comparison to pilot reliability.

**Outputs:**
- `outputs/study1_analysis/tables/validation_kappa.csv`
- `outputs/study1_analysis/tables/confusion_matrix_validation.csv`
- `outputs/study1_analysis/figures/validation_kappa_per_category.png`
- `outputs/study1_analysis/figures/confusion_matrix_validation.png`

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
from study1_helpers import *

traces = load_traces()
manual = load_manual_traces()
df = build_sentence_df(traces)

## 3a. Validation κ Results

In [ ]:
section_header('3a. Validation Kappa Results')

kappa_df = load_kappa_results()
display(kappa_df[['trace', 'n_sentences', 'kappa_micro', 'kappa_macro', 'agreement_pct']])

# Compute summary statistics
print(f'\nSummary across {len(kappa_df)} validation traces:')
for col in ['kappa_micro', 'kappa_macro', 'agreement_pct']:
    vals = kappa_df[col]
    print(f'  {col}: mean={vals.mean():.3f}, median={vals.median():.3f}, SD={vals.std():.3f}')

# Save with added means
summary_row = {'trace': 'MEAN'}
for col in kappa_df.columns:
    if col == 'trace':
        continue
    try:
        summary_row[col] = kappa_df[col].mean()
    except TypeError:
        summary_row[col] = ''
kappa_out = pd.concat([kappa_df, pd.DataFrame([summary_row])], ignore_index=True)
save_table(kappa_out, 'validation_kappa.csv')

## 3b. Per-Category One-vs-Rest κ

In [ ]:
section_header('3b. Per-Category One-vs-Rest Kappa')

# Extract per-category kappa columns
cat_cols = [f'kappa_{m}' for m in MICRO_LABELS]
cat_kappas = {}
for m in MICRO_LABELS:
    col = f'kappa_{m}'
    if col in kappa_df.columns:
        cat_kappas[m] = kappa_df[col].mean()
    else:
        cat_kappas[m] = np.nan

cat_df = pd.DataFrame([
    {'category': m, 'mean_kappa': cat_kappas[m],
     'gate': 'PASS' if cat_kappas[m] >= 0.50 else 'FAIL'}
    for m in MICRO_LABELS
])
print('Per-category validation κ (mean across 7 traces):')
display(cat_df)

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#55A868' if k >= 0.50 else '#C44E52' for k in cat_df['mean_kappa']]
ax.bar(cat_df['category'], cat_df['mean_kappa'], color=colours, edgecolor='white')
ax.axhline(y=0.50, color='black', linestyle='--', linewidth=1, label='Threshold (κ=0.50)')
ax.set_ylabel('Mean Cohen\'s κ')
ax.set_xlabel('Micro-label')
ax.set_title('Per-Category Validation κ (One-vs-Rest)')
ax.legend()
ax.set_xticklabels(MICRO_LABELS, rotation=45, ha='right')
ax.set_ylim(0, 1.05)
plt.tight_layout()
save_fig(fig, 'validation_kappa_per_category.png')
plt.show()

## 3c. Confusion Analysis on Validation Traces

In [ ]:
section_header('3c. Confusion Matrix')

# Match auto-coded and manual-coded sentences for validation traces
pairs = get_validation_pairs(traces, manual)
print(f'Matched {len(pairs)} validation trace pairs')

all_manual = []
all_auto = []
for key, a_df, m_df in pairs:
    # Merge on sentence_id
    merged = a_df[['sentence_id', 'micro_label']].merge(
        m_df[['sentence_id', 'micro_label']],
        on='sentence_id', suffixes=('_auto', '_manual')
    )
    all_manual.extend(merged['micro_label_manual'].tolist())
    all_auto.extend(merged['micro_label_auto'].tolist())

print(f'Total matched sentences: {len(all_manual)}')

# Compute confusion matrix
cm = confusion_matrix(all_manual, all_auto, labels=MICRO_LABELS)
cm_df = pd.DataFrame(cm, index=MICRO_LABELS, columns=MICRO_LABELS)

# Row-normalise for heatmap
cm_norm = cm_df.div(cm_df.sum(axis=1), axis=0).fillna(0)

print('\nConfusion matrix (rows=manual, cols=auto):')
display(cm_df)

# Save
save_table(cm_df, 'confusion_matrix_validation.csv')

# Visualise heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=MICRO_LABELS, yticklabels=MICRO_LABELS,
            linewidths=0.5, vmin=0, vmax=1)
ax.set_xlabel('Auto-coded label')
ax.set_ylabel('Manual label')
ax.set_title('Validation Confusion Matrix (row-normalised)')
plt.tight_layout()
save_fig(fig, 'confusion_matrix_validation.png')
plt.show()

## 3d. Known Issues

In [ ]:
section_header('3d. Known Issues')

# Flag trace_019 and trace_004 if they show the documented patterns
for _, row in kappa_df.iterrows():
    trace = row['trace']
    if 'trace_019' in trace:
        print(f"trace_019: κ_micro={row['kappa_micro']:.3f}, κ_macro={row['kappa_macro']:.3f}")
        if row['kappa_micro'] < 0.50:
            print('  → LOW κ: Rapid hypothesis cycling compresses HYPO/TEST/JUDGE boundaries.')
            print(f'  → Macro κ = {row["kappa_macro"]:.3f} confirms INVESTIGATE macro-state detected.')
        else:
            print('  → κ above concern threshold in current CSV.')
    if 'trace_004' in trace:
        print(f"\ntrace_004: κ_micro={row['kappa_micro']:.3f}, κ_macro={row['kappa_macro']:.3f}")
        if row['kappa_micro'] > row['kappa_macro']:
            print('  → ANOMALY: micro > macro. Caused by TEST→SYNTHESIZE swaps crossing')
            print('    INVESTIGATE/OBSERVE macro boundary. Skewed marginals inflate chance-agreement')
            print('    correction in κ — a statistical artefact, not a coding quality issue.')
        else:
            print('  → No micro > macro anomaly in current CSV.')

## 3e. Comparison to Pilot Reliability

In [ ]:
section_header('3e. Comparison to Pilot Reliability')

comparison = pd.DataFrame([
    {'metric': 'Pilot Tax A inter-coder κ', 'value': 0.515, 'type': 'Inter-coder (Sonnet vs GPT-4o)'},
    {'metric': 'Pilot Tax B inter-coder κ', 'value': 0.401, 'type': 'Inter-coder (Sonnet vs GPT-4o)'},
    {'metric': 'Phase 3 micro validation κ', 'value': kappa_df['kappa_micro'].mean(), 'type': 'Expert validation (auto vs manual)'},
    {'metric': 'Phase 3 macro validation κ', 'value': kappa_df['kappa_macro'].mean(), 'type': 'Expert validation (auto vs manual)'},
])
display(comparison)
print('\nNote: These are different metrics (inter-coder vs expert validation),')
print('so direct comparison is contextual, not statistical. The improvement reflects')
print('both better taxonomy definitions and the shift from "do two auto-coders agree?"')
print('to "does the auto-coder agree with the expert?".')